# PiCar-X camera challenge: build your driver checkpoint by checkpoint

Your camera gives you a word. Your Python code decides what the car should do.

**How to work:**

1. Run the collapsed setup cell once.
2. Write all your driving logic in **one student code cell** below. Follow its numbered checkpoint comments.
3. Run your code cell, then run the **debug cell directly underneath it**.
4. Read the first failure, change your code, and run those same two cells again.

Save a copy in Google Drive if you use Colab. No installs, camera, or hardware are needed. The camera is a mock: the tests supply pretend readings, with no computer vision code to learn.

## Your command reference

| Code | Meaning |
| --- | --- |
| `car.scan()` | Returns `"clear"`, `"human"`, or `"stop sign"` |
| `car.ultrasonic.read()` | Returns the distance ahead in centimeters |
| `car.stop()` | Requests a stop |
| `car.forward(30)` | Requests movement at speed 30 |
| `car.set_dir_servo_angle(-25)` | Steers left; use `25` for right and `0` for straight |

Use `=` to save a reading and `==` to compare values. Put words in quotes. Indent the lines inside an `if`. Use `return` by itself to finish this function after handling a hazard.

`"clear"` only means the camera did not report a tracked person or sign. There can still be a box nearby. These are decision checks; the full driving route will also need steering and stop timing.
## Make it yours: checkpoint 7

Name your driver, invent an obstacle (a giant sneaker, a pizza box, a sleeping dragon), and choose **left** or **right**. Change the three settings at the top of your student cell.

Your mock camera will report your object's name. Add a rule that steers toward your chosen side and moves forward. When the camera says `"clear"` again, straighten the wheels. If the object is closer than 25 cm, stop instead.

This practices **starting a detour and returning to straight driving**. The mock has no road geometry and cannot prove the car went around an object. New object names are pretend labels here; adding one does not train the real camera detector or add it to the full simulator.


In [ ]:
# @title 1. Start the pretend car (run once)
class MockPicarx:
    """A pretend sensor reading and a record of requested actions; no real motors."""
    def __init__(self, label="clear", distance=100):
        self.label = label
        self.distance = distance
        self.action = "stop"
        self.steering = 0
        self.history = []
        self.ultrasonic = self

    def scan(self) -> str:
        return self.label

    def read(self) -> float:
        return self.distance

    def forward(self, speed: float) -> None:
        if not 0 <= speed <= 100:
            raise ValueError("Use a speed between 0 and 100.")
        self.action = "forward" if speed > 0 else "stop"
        self.history.append((self.action, speed))

    def stop(self) -> None:
        self.action = "stop"
        self.history.append(("stop", 0))

    def set_dir_servo_angle(self, angle: float) -> None:
        if not -30 <= angle <= 30:
            raise ValueError("Steering must be between -30 and 30 degrees.")
        self.steering = angle


def check_decisions(function):
    cases = [
        ("clear", "forward", "An empty view should allow movement."),
        ("human", "stop", "Yield while a person is visible."),
        ("stop sign", "stop", "Begin stopping when a sign is seen."),
        ("unknown", "stop", "An unfamiliar reading should not mean go."),
    ]
    passed = 0
    for label, expected, hint in cases:
        try:
            actual = function(label)
            ok = actual == expected
            passed += ok
            print(f"{'PASS' if ok else 'TRY AGAIN'} | {label}: {actual!r}")
            if not ok:
                print("  Hint:", hint, "Return", repr(expected))
        except Exception as error:
            print(f"TRY AGAIN | {label}: {type(error).__name__}: {error}")
    print(f"Decision checks: {passed}/{len(cases)}")


def play_scenario(controller, frames, title):
    car = MockPicarx()
    memory = {}
    print("\nSCENARIO:", title)
    for step, (label, distance) in enumerate(frames, start=1):
        car.label, car.distance = label, distance
        car.history = []
        try:
            controller(car, memory)
        except Exception as error:
            print(f"Step {step}: controller error: {error}")
            return False
        personal_object = globals().get("custom_object", "")
        detour = label == personal_object and distance >= 25
        must_stop = (label != "clear" and not detour) or distance < 25
        unsafe_command = any(action == "forward" for action, speed in car.history) if must_stop else False
        correct = car.action == ("stop" if must_stop else "forward") and not unsafe_command
        if correct and not must_stop:
            expected_angle = (-25 if globals().get("avoid_direction") == "left" else 25) if detour else 0
            if car.steering != expected_angle:
                print(f"TRY AGAIN: steering is {car.steering}; expected {expected_angle}.")
                print("Use your chosen side for the custom object; straighten when the view clears.")
                return False
        print(f"Step {step} | camera={label!r}, distance={distance} cm -> {car.action}, steering={car.steering}")
        if not correct:
            print("TRY AGAIN:", "Stop without issuing a forward command for this hazard." if must_stop
                  else "The view is clear and the object is far away. Let the car move.")
            return False
    print("PASS: all decisions matched this scenario.")
    return True

print("Pretend car ready. No camera or motors are connected.")

## Student code — edit this cell

Start with checkpoint 1. Add one rule at a time. Keep all your work here so you can see how the rules fit together.

`control_step()` handles one moment. The runner repeats it for you. Do not add an endless loop or a delay. Leave `memory` unused for now.

The starter stops everywhere, so some checks will pass before you write anything. Your goal is to explain and pass **every** checkpoint, including moving again after a person leaves.

In [ ]:
# MAKE IT YOURS — edit these three values.
driver_name = "Captain Cautious"
custom_object = "giant sneaker"
avoid_direction = "left"  # Use "left" or "right".


def control_step(car, memory: dict) -> None:
    """Read the sensors and choose one driving action."""

    # CHECKPOINT 1 — READ THE ROAD
    # Save car.scan() in a variable named scan_input.
    # Save car.ultrasonic.read() in a variable named distance.
    # TODO: write your two sensor-reading lines here.

    # CHECKPOINT 2 — PERSON CROSSING
    # If scan_input is "human", stop and finish this step.
    # Tip: call car.stop(), then use return on the next indented line.
    # TODO: write your person rule here.

    # CHECKPOINT 3 — STOP SIGN
    # If scan_input is "stop sign", stop and finish this step.
    # This practice checks stopping when the sign appears, not when to leave.
    # TODO: write your stop-sign rule here.

    # CHECKPOINT 4 — BOX AHEAD
    # Even a "clear" camera can miss a box. If distance < 25, stop and return.
    # Exactly 25 is allowed in this practice; 24 is too close.
    # TODO: write your distance rule here.

    # CHECKPOINT 7 — YOUR PERSONAL OBSTACLE (add this after checkpoints 1–6 work)
    # This rule belongs BEFORE the unknown-word rule, even though it is task 7.
    # If scan_input == custom_object, choose your steering using avoid_direction:
    # "left" -> car.set_dir_servo_angle(-25); "right" -> angle 25.
    # Then call car.forward(30) and return.
    # Keep checkpoint 4 ABOVE this rule so a nearby object still causes a stop.
    # TODO: write your custom obstacle rule here.

    # CHECKPOINT 5 — UNEXPECTED CAMERA WORD
    # If scan_input is not "clear", stop and return.
    # Tip: != means "is not equal to". Do not trust an unfamiliar reading.
    # TODO: write your unexpected-reading rule here.

    # CHECKPOINT 6 — GO WHEN IT IS CLEAR
    # If no earlier rule stopped the car, straighten with car.set_dir_servo_angle(0),
    # then request car.forward(30). This resets steering after your detour.
    # Replace the starter stop below when your safety rules are ready.
    car.stop()


# OPTIONAL: change this sequence to invent your own test.
# Each pair means (camera word, distance in centimeters).
my_test = [("clear", 100), (custom_object, 70), (custom_object, 10), ("clear", 100)]
print("Student code loaded. Now run the debug cell below.")

In [ ]:
# DEBUG — run after running your student code cell. Do not edit this cell.
# Each checkpoint starts with a fresh mock car and fresh memory.
settings_ok = (
    isinstance(driver_name, str) and bool(driver_name.strip())
    and isinstance(custom_object, str) and bool(custom_object.strip())
    and custom_object == custom_object.strip()
    and custom_object not in ("clear", "human", "stop sign", "unknown")
    and avoid_direction in ("left", "right")
)
if not settings_ok:
    raise ValueError("Give your driver and object names; use a new object label without outer spaces, and choose left or right.")
print(f"DRIVER: {driver_name} | Personal obstacle: {custom_object} | Detour: {avoid_direction}")
checkpoint_tests = [
    ("2: Person crossing", [("human", 80)],
     "Compare scan_input to 'human', stop, and return before any forward command."),
    ("3: Stop sign", [("stop sign", 80)],
     "Compare scan_input to 'stop sign', stop, and return."),
    ("4: Box ahead + distance boundary", [("clear", 25), ("clear", 24)],
     "Read distance in checkpoint 1. Stop below 25; allow exactly 25 if the view is clear."),
    ("5: Unexpected word", [("unknown", 100)],
     "An unfamiliar camera word should cause a stop, never a forward command."),
    ("6: Clear road", [("clear", 100), ("clear", 60)],
     "After checking hazards, call car.forward(30). Remove the starter stop at the bottom."),
    ("7: Your obstacle + straighten afterward", [(custom_object, 70), ("clear", 100)],
     "Put your custom-object rule before checkpoint 5. Steer -25 left or 25 right, move, then return. Straighten on clear."),
    ("7: Your obstacle too close", [(custom_object, 20)],
     "Checkpoint 4 must stop nearby objects before your custom rule can move the car."),
    ("Together: wait, then move again", [("clear", 100), ("human", 80), ("human", 70), ("clear", 100)],
     "Read both sensors on every call. Stop while a person is visible; move when the road clears."),
]
passed = 0
for title, frames, hint in checkpoint_tests:
    if play_scenario(control_step, frames, title):
        passed += 1
    else:
        print("WHERE TO LOOK:", hint)
print(f"\nCHECKPOINT RESULTS: {passed}/{len(checkpoint_tests)} checks passed.")
print("Checkpoint 1 is used by all the rules: make sure you actually read both sensors.")
print("A passing check tests behavior; be ready to explain the rule in your code.")
play_scenario(control_step, my_test, "Your custom sequence")
print("\nAfter each edit: run STUDENT CODE first, then DEBUG. If the code cell has a red error, fix it before debugging.")

## Explain your driver

- Which rule prevents a box collision when the camera says `"clear"`?
- What would happen if you called `car.forward(30)` before checking for a person?
- Why does `return` help after stopping?
- Change `my_test` in your student cell. Predict each action, then run both cells again.

- Why did you choose your personal object and driver name?
- Why must the distance rule run before your custom-object rule?
- Why must your custom-object rule run before the unexpected-word rule?

**Next:** copy your `control_step()` function into `student_controller.py` or the student-code cell of the full driving notebook. Keep the existing `CoursePicarx` import there. Do not copy the mock setup or debug cells. Your personal rule also uses the
`custom_object` and `avoid_direction` settings: keep them with your function.
The full simulator currently reports only its built-in labels, so the custom-object
branch is notebook practice until that environment supports your chosen object.

This notebook checks your first decision layer, not a complete route or the capstone score. Stopping at a visible sign is only the beginning: the full route also needs a measured stop duration, a decision about leaving the sign, steering, and obstacle navigation. A notebook step is not one real second. Physical camera readings will need calibration and can be wrong.